In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **imports**

In [ ]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

!pip install youtube-transcript-api

!pip install -U langchain langchain-community langchain-core transformers==4.52.4
!pip install sentence-transformers

# **Model Config**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

# **Schema Config**

In [ ]:
sum_schema = ResponseSchema(
    name="summary",
    description="A concise and accurate summary of the YouTube video transcript, including the main topic, key points, and important conclusions."
)

response_schemas = [sum_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

sum_extraction_template = """
You are an expert at summarizing YouTube video transcripts.

Your task is to produce a concise, factual summary of the transcript.

Instructions:
- Identify the video's main topic.
- Summarize the most important ideas and key points.
- Include any significant conclusions or recommendations.
- Ignore greetings, sponsorships, advertisements, repeated phrases, and outros.
- Do not invent or infer information that is not present in the transcript.
- If the transcript is incomplete, summarize only the available content.

Return your response ONLY in the following format:
{format_instructions}

Transcript:
{user_input}
"""

# **Helper function**

In [ ]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        return f"```json\n{matches[-1]}\n```"

    return text

# **YT Transcript**

In [ ]:
from urllib.parse import urlparse, parse_qs
import re

def extract_video_id(url: str) -> str | None:
    """Extract the YouTube video ID from common URL formats."""
    parsed = urlparse(url)

    if parsed.hostname in ("youtu.be", "www.youtu.be"):
        return parsed.path.lstrip("/")

    if parsed.hostname in (
        "youtube.com",
        "www.youtube.com",
        "m.youtube.com",
    ):
        if parsed.path == "/watch":
            return parse_qs(parsed.query).get("v", [None])[0]

        if parsed.path.startswith("/shorts/"):
            return parsed.path.split("/")[2]

        if parsed.path.startswith("/embed/"):
            return parsed.path.split("/")[2]

    # Fallback regex
    match = re.search(r"(?:v=|/)([0-9A-Za-z_-]{11})", url)
    return match.group(1) if match else None

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

def get_transcript(youtube_url: str) -> str:
    video_id = extract_video_id(youtube_url)

    if not video_id:
        return "Unable to extract YouTube video ID."

    try:
        transcript = ytt_api.fetch(video_id)

        return "\n".join(
            snippet.text
            for snippet in transcript
        )

    except Exception as e:
        return f"Transcript unavailable: {e}"

# **Add api keys**

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
TEMP_API_URL = user_secrets.get_secret("TEMP_API_URL")
API_KEY = user_secrets.get_secret("API_KEY")

# **Endpoint Setup**

In [ ]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/api/v1/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")

    data = await req.json()

    youtube_url = data.get("youtube_url", "")
    max_len = data.get("max_length", 500)

    transcript = get_transcript(youtube_url) if youtube_url else ""

    if not transcript:
        raise HTTPException(
            status_code=400,
            detail="Unable to retrieve transcript"
        )

    prompt = PromptTemplate(
        template=sum_extraction_template,
        input_variables=[
            "user_input",
            "format_instructions"
        ]
    ).format(
        user_input=transcript,
        format_instructions=format_instructions
    )

    response = generate_text(prompt, max_length=max_len)

    if isinstance(response, (tuple, list)):
        response = response[0]

    json_text = extract_json_block(response)

    try:
        parsed_output = output_parser.parse(json_text)
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Failed to parse model output: {str(e)}"
        )

    return {
        "response": parsed_output,
        "source": {
            "youtube_url": youtube_url
        }
    }

# **Ngrok Port Setup**

In [ ]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)
def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

# **Test API request**

In [ ]:
import requests

URL = f"{TEMP_API_URL}/api/v1/generate"  
headers = {"Authorization": f"Bearer {API_KEY}"}
payload = {
    "youtube_url": "https://youtu.be/jv6hBaCtsFg",
    "max_length": 3000
}

In [ ]:
res = requests.post(URL, headers=headers, json=payload)

In [ ]:
ans = res.json()["response"]["summary"]

In [ ]:
print(ans)